In [ ]:
# Connect to source data
!gdown --id 1OMSQFQ4sMYxJhotprfVcQ0mPB222WqGP
!unzip -q super-ai-engineer-ss-6-sleep-stage-classification.zip

In [ ]:
# === CONFIGURATION ===
DATA_DIR = "/content"
TR_DIR = f"{DATA_DIR}/train/train"
TEST_DIR = f"{DATA_DIR}/test_segment/test_segment"



In [ ]:
import os
import pandas as pd

DATA_DIR = "/content" # เช็กให้ชัวร์ว่าโฟลเดอร์ใน Colab ชื่อนี้ไหม
TEST_DIR = f"{DATA_DIR}/test_segment/test_segment"

sub_df = pd.read_csv("sample_submission.csv")

# ลองปริ้นท์ Path ของไฟล์แรกดูว่าหน้าตาเป็นยังไง และมันมีอยู่จริงไหม
test_id = sub_df['id'].iloc[0] # "test001_00000"
folder_name = f"test{test_id[4:7]}"
file_path = os.path.join(TEST_DIR, folder_name, f"{test_id}.csv")

print("Expected Path:", file_path)
print("File Exists?:", os.path.exists(file_path))

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from scipy import stats

SEGMENT_SIZE = 480

# ลิสต์สำหรับเก็บข้อมูลฟีเจอร์และฉลากคำตอบ
X_list = []
y_list = []
# ลิสต์เก็บกลุ่มของข้อมูลเพื่อใช้ทำ GroupKFold ในภายหลัง
groups_list = []

# ดึงรายชื่อไฟล์ทั้งหมดที่ขึ้นต้นด้วย train และลงท้ายด้วย .csv
file_pattern = os.path.join(TR_DIR, "train*.csv")
train_files = sorted(glob.glob(file_pattern))

print(f"Found {len(train_files)} training files.")

for file_path in train_files:
    # ดึงหมายเลข subject จากชื่อไฟล์ เช่น train001.csv -> 001
    file_name = os.path.basename(file_path)
    subject_id = file_name.replace('train', '').replace('.csv', '')

    # อ่านไฟล์ CSV
    df = pd.read_csv(file_path)

    # แยกคอลัมน์ Feature และ Label
    feature_cols = [col for col in df.columns if col != 'Sleep_Stage']

    # จำนวนแถวทั้งหมดในไฟล์นี้
    n_rows = len(df)

    # หั่นข้อมูลทีละ 480 แถว
    for start_idx in range(0, n_rows, SEGMENT_SIZE):
        end_idx = start_idx + SEGMENT_SIZE

        # ตัดข้อมูลมา 1 ท่อน
        segment = df.iloc[start_idx:end_idx]

        # ตรวจสอบว่าท่อนนี้มีข้อมูลครบ 480 แถวหรือไม่ (ป้องกันข้อมูลเศษตอนท้ายไฟล์)
        if len(segment) == SEGMENT_SIZE:
            # ดึงเฉพาะค่าฟีเจอร์ แปลงเป็น numpy array
            features = segment[feature_cols].values

            # ดึงฉลากคำตอบ โดยหาค่าที่โผล่มาบ่อยที่สุด (Mode) ใน 480 แถวนี้
            labels = segment['Sleep_Stage'].values
            # Use np.unique to find the mode for non-numeric labels
            unique_labels, counts = np.unique(labels, return_counts=True)
            segment_label = unique_labels[np.argmax(counts)]

            # เก็บข้อมูลลงลิสต์
            X_list.append(features)
            y_list.append(segment_label)
            groups_list.append(subject_id)

# แปลงลิสต์ทั้งหมดให้เป็น Numpy Array เพื่อนำไปใช้งานต่อ
X = np.array(X_list)
y = np.array(y_list)
groups = np.array(groups_list)

print("Segmentation Completed!")
print(f"Shape of X (Features): {X.shape}")
print(f"Shape of y (Labels): {y.shape}")
print(f"Shape of groups: {groups.shape}")

# ตรวจสอบจำนวนข้อมูลในแต่ละคลาสเพื่อดู Class Imbalance
unique, counts = np.unique(y, return_counts=True)
class_distribution = dict(zip(unique, counts))
print(f"Class Distribution: {class_distribution}")

In [ ]:
!pip install lightgbm
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import skew, kurtosis

In [ ]:
def extract_features(X_data):
    # X_data shape: (N_segments, 480_timesteps, 8_channels)
    features_list = []

    for i in range(X_data.shape[0]):
        segment = X_data[i] # shape: (480, 8)

        # คำนวณค่าสถิติของแต่ละช่องสัญญาณ (Channel)
        mean_val = np.mean(segment, axis=0)
        std_val = np.std(segment, axis=0)
        min_val = np.min(segment, axis=0)
        max_val = np.max(segment, axis=0)
        skew_val = skew(segment, axis=0)
        kurt_val = kurtosis(segment, axis=0)

        # นำค่าสถิติมาต่อกันเป็นเวกเตอร์ยาวๆ 1 เส้น (8 * 6 = 48 ฟีเจอร์)
        combined_features = np.concatenate([
            mean_val, std_val, min_val, max_val, skew_val, kurt_val
        ])

        features_list.append(combined_features)

    return np.array(features_list)

print("Extracting features from Train data...")
# ใช้ตัวแปร X ดิบรอบแรกสุด ที่ยังไม่ได้ scale เลยนะครับ (Tree-model ไม่ต้อง scale ข้อมูล)
X_features = extract_features(X)
print("Extracted Features Shape:", X_features.shape)



In [ ]:
# แปลง Label (ถ้ายังไม่ได้แปลง)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

gkf = GroupKFold(n_splits=5)
oof_preds = np.zeros(len(X_features))
f1_scores = []

# ใช้ LightGBM Classifier
clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [ ]:
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_features, y_encoded, groups)):
    print(f"--- Fold {fold + 1} ---")

    X_tr, y_tr = X_features[train_idx], y_encoded[train_idx]
    X_va, y_va = X_features[val_idx], y_encoded[val_idx]

    # เทรนโมเดล
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # ทำนายและวัดผล
    val_preds = clf.predict(X_va)
    score = f1_score(y_va, val_preds, average='weighted')
    f1_scores.append(score)
    print(f"Fold {fold + 1} Weighted F1-Score: {score:.4f}")

print(f"\nAverage F1-Score: {np.mean(f1_scores):.4f}")

In [ ]:
import os
import pandas as pd

TEST_DIR = f"{DATA_DIR}/test_segment/test_segment"
sub_df = pd.read_csv("sample_submission.csv")
predictions = []

print("Starting Inference with LightGBM...")

for test_id in sub_df['id']:
    folder_name = f"test{test_id[4:7]}"
    file_path = os.path.join(TEST_DIR, folder_name, f"{test_id}.csv")

    if not os.path.exists(file_path):
        predictions.append("W")
        continue

    test_df = pd.read_csv(file_path)
    features_raw = test_df[['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']].values

    if len(features_raw) == 480:
        # 1. ทำเป็น 3D array ชั่วคราวเพื่อเข้าฟังก์ชัน extract
        features_reshaped = np.expand_dims(features_raw, axis=0)

        # 2. สกัดฟีเจอร์ จะได้ shape (1, 48)
        feat_extracted = extract_features(features_reshaped)

        # 3. ทำนายผล
        pred_idx = clf.predict(feat_extracted)[0]
        pred_label = le.inverse_transform([pred_idx])[0]
        predictions.append(pred_label)
    else:
        predictions.append("N2")

sub_df['labels'] = predictions
sub_df.to_csv("lightgbm_submission.csv", index=False)
print("Saved lightgbm_submission.csv")